In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

# ── 데이터 로드 ──────────────────────────────────────────
df = pd.read_csv('./final_underwriting_governance_data.csv')
print(f"데이터: {len(df)}건")

# ── 에이전트별 판정 파싱 ─────────────────────────────────
agents = ['수석 언더라이터', '임상 전문의', '보험 계리사', '예방의학 전문가', '준법 감시인']

def parse_grades(text):
    if not isinstance(text, str):
        return []
    grades = []
    for agent in agents:
        if agent in text:
            idx = text.find(agent)
            snippet = text[idx:idx+60]
            m = re.search(r'(Standard|Loading|Decline|Error)', snippet)
            if m and m.group(1) != 'Error':
                grades.append(m.group(1))
            else:
                grades.append(None)
        else:
            grades.append(None)
    return grades

df['판정목록'] = df['전문가_통합_심사평'].apply(parse_grades)
df['유효판정'] = df['판정목록'].apply(lambda x: [g for g in x if g is not None])
print(f"유효 판정 보유 건수: {len(df[df['유효판정'].apply(len) >= 3])}건")

# ── Cr 및 최종판정 계산 함수 ─────────────────────────────
def calc_cr_and_grade(grades):
    if not grades:
        return None, None
    final = max(set(grades), key=grades.count)
    cr = grades.count(final) / len(grades)
    return cr, final

# ── GRI 계산 함수 ─────────────────────────────────────────
grade_map = {'Standard': 0, 'Loading': 1, 'Decline': 2}

def calc_gri(sf, cr, rule_grade, ai_grade):
    f_risk = 1 - sf
    u_risk = 1 - cr
    try:
        conflict = abs(grade_map.get(rule_grade, 1) - grade_map.get(ai_grade, 1)) * 0.5
    except:
        conflict = 0
    return 0.4 * f_risk + 0.4 * u_risk + 0.2 * conflict

# ── 에이전트 수별 시뮬레이션 ─────────────────────────────
np.random.seed(42)
N_BOOT = 1000  # 7인 부트스트랩 반복 수

# governance_audit_full에서 Sf, Rule_판정 가져오기
df_gov = pd.read_csv('governance_audit_full.csv', encoding='utf-8-sig')
df_merged = df.merge(df_gov[['개인아이디', 'Sf_Score', 'Rule_판정']], on='개인아이디', how='inner')
df_valid = df_merged[df_merged['유효판정'].apply(len) >= 3].copy()
print(f"분석 대상: {len(df_valid)}건")

results = {}

for n_agents in [3, 5]:
    crs, finals, gris, l4s = [], [], [], []
    for _, row in df_valid.iterrows():
        grades = row['유효판정']
        # n_agents만큼 비복원 추출
        selected = grades[:n_agents] if len(grades) >= n_agents else grades
        cr, final = calc_cr_and_grade(selected)
        if cr is None:
            continue
        gri = calc_gri(row['Sf_Score'], cr, row['Rule_판정'], final)
        crs.append(cr)
        finals.append(final)
        gris.append(gri)
        l4s.append(1 if gri >= 0.18 else 0)

    results[n_agents] = {
        'Cr=1.0 비율(%)': round(sum(1 for c in crs if c == 1.0) / len(crs) * 100, 1),
        'Cr<1.0 비율(%)': round(sum(1 for c in crs if c < 1.0) / len(crs) * 100, 1),
        '평균 Cr': round(np.mean(crs), 4),
        '평균 GRI': round(np.mean(gris), 4),
        'L4 비율(%)': round(np.mean(l4s) * 100, 1),
        'N': len(crs)
    }

# 7인: 복원추출 부트스트랩
crs7, finals7, gris7, l4s7 = [], [], [], []
for _, row in df_valid.iterrows():
    grades = row['유효판정']
    boot_grades = list(np.random.choice(grades, size=7, replace=True))
    cr, final = calc_cr_and_grade(boot_grades)
    if cr is None:
        continue
    gri = calc_gri(row['Sf_Score'], cr, row['Rule_판정'], final)
    crs7.append(cr)
    finals7.append(final)
    gris7.append(gri)
    l4s7.append(1 if gri >= 0.18 else 0)

results[7] = {
    'Cr=1.0 비율(%)': round(sum(1 for c in crs7 if c == 1.0) / len(crs7) * 100, 1),
    'Cr<1.0 비율(%)': round(sum(1 for c in crs7 if c < 1.0) / len(crs7) * 100, 1),
    '평균 Cr': round(np.mean(crs7), 4),
    '평균 GRI': round(np.mean(gris7), 4),
    'L4 비율(%)': round(np.mean(l4s7) * 100, 1),
    'N': len(crs7)
}

# ── 결과 출력 ────────────────────────────────────────────
print("\n" + "=" * 70)
print("에이전트 수별 민감도 분석 결과")
print("=" * 70)
print(f"{'지표':<20} {'3인':>10} {'5인(실제)':>12} {'7인(부트)':>12}")
print("-" * 70)

metrics = ['Cr=1.0 비율(%)', 'Cr<1.0 비율(%)', '평균 Cr', '평균 GRI', 'L4 비율(%)']
for m in metrics:
    print(f"{m:<20} {results[3][m]:>10} {results[5][m]:>12} {results[7][m]:>12}")

print(f"\n주: 7인은 5인 판정의 복원추출 부트스트랩(독립성 동일 가정 하 시뮬레이션)")
print(f"    실제 독립적 7인 에이전트 실험과는 구별되어야 함")

pd.DataFrame(results).T.to_csv('agent_sensitivity_result.csv', encoding='utf-8-sig')
print("\n결과 저장: agent_sensitivity_result.csv")

데이터: 10058건
유효 판정 보유 건수: 4500건
분석 대상: 4500건

에이전트 수별 민감도 분석 결과
지표                           3인       5인(실제)       7인(부트)
----------------------------------------------------------------------
Cr=1.0 비율(%)               83.0         74.0         76.8
Cr<1.0 비율(%)               17.0         26.0         23.2
평균 Cr                    0.9434       0.9238       0.9321
평균 GRI                   0.0654       0.0725       0.0689
L4 비율(%)                    8.8          7.4          8.7

주: 7인은 5인 판정의 복원추출 부트스트랩(독립성 동일 가정 하 시뮬레이션)
    실제 독립적 7인 에이전트 실험과는 구별되어야 함

결과 저장: agent_sensitivity_result.csv
